# 1. Business Understanding and Analytic Approach

**Đề tài:** *Zero-Shot Foundation Models versus Trained Learners for Weekly Street-Tree Service Demand: Evidence from San Francisco 311 Across an Intake-System Change*


## 1. Giới thiệu bài toán

Cây xanh đường phố được quản lý như một hạng mục hạ tầng đô thị: cắt tỉa, xử lý cây gãy đổ sau bão, loại bỏ cây nguy hiểm đều do đội bảo trì của thành phố lên lịch. San Francisco công khai cả nhật ký yêu cầu bảo trì (311 Cases) lẫn danh mục cây đường phố (Street Tree List), nhưng phần lớn việc điều phối đội bảo trì vẫn dựa vào kinh nghiệm và tồn đọng công việc, chứ chưa dựa trên dự báo nhu cầu theo khu vực và thời gian — khiến các tuần có bão dễ bị quá tải so với năng lực đội ngũ bình thường.

Dự án này khai thác dữ liệu mở của San Francisco 311 Cases, Street Tree List, và SF Street-Use Permits (dùng làm nguồn phụ ngoại sinh chính) — cùng dữ liệu thời tiết NOAA (dùng mở rộng) — nhằm:

- **Định vị bức tranh nhu cầu**: xác định khu phố, mùa và điều kiện nào có nhu cầu bảo trì cây cao nhất, cao hơn bao nhiêu lần so với khu phố thấp nhất, và nhu cầu ghi nhận thay đổi ra sao khi hệ thống tiếp nhận 311 đổi bộ phân loại yêu cầu vào tháng 6/2024.
- **Định lượng tác động của hoạt động đô thị**: đánh giá hoạt động thi công đường phố (số giấy phép đang hoạt động) và điều kiện thời tiết ảnh hưởng ra sao đến tần suất yêu cầu bảo trì, bằng kiểm định thống kê chứ không chỉ quan sát định tính.
- **Xây dựng nền tảng dự báo đáng tin cậy**: so sánh nhiều họ mô hình — từ mốc đơn giản, mô hình đếm, hồi quy tuyến tính, cây tăng cường, đến các mô hình nền chuỗi thời gian zero-shot — để dự báo nhu cầu tuần tới cho từng khu phố, đồng thời đánh giá độ tin cậy của dự báo và độ bền của từng mô hình khi cách ghi nhận dữ liệu thay đổi giữa chừng.

## 2. Đóng góp của bài

1. **Một bộ đánh giá công bằng, chống rò rỉ dữ liệu theo thời gian** trên dữ liệu mở của San Francisco: 41 khu phố × 614 tuần (2015–2026). Bộ đánh giá so sánh các mốc đơn giản, mô hình đếm, mô hình học máy đã tinh chỉnh và nhiều mô hình nền tảng zero-shot. Tất cả cùng chạy trên nhiều mốc thời gian.
2. **Phân tích điểm gãy tháng 6/2024** của hệ thống 311: ghép bộ phân loại cũ và mới, đo mức thay đổi, và đánh giá độ bền của từng họ mô hình trước và sau điểm gãy. Tập kiểm tra trải qua điểm gãy này nên cho phép quan sát trực tiếp cách mỗi mô hình phản ứng khi cách ghi nhận dữ liệu thay đổi.
3. **Đánh giá độ tin cậy vận hành:** so sánh khoảng dự báo có sẵn của mô hình nền tảng với khoảng dự báo conformal, và thử cơ chế cảnh báo tuần cao điểm cho từng khu phố.
4. **Mã nguồn có thể chạy lại hoàn toàn** trên bản dữ liệu đã lưu, cùng quy trình tải dữ liệu từ API công khai của thành phố.

## 3. Câu hỏi nghiên cứu

| | Câu hỏi | Notebook |
|---|---|---|
| **RQ1** (Cấu trúc nhu cầu) | Nhu cầu bảo trì cây biến động thế nào theo khu phố, theo mùa, qua các năm, và điểm gãy tháng 6/2024 làm thay đổi dữ liệu đến mức nào? | 3, 4, 6 |
| **RQ2** (So sánh mô hình) | Mô hình nền tảng zero-shot có dự báo tuần tới chính xác hơn mô hình huấn luyện riêng không, khi đánh giá trên nhiều mốc thời gian và trên cả khu phố ít lẫn nhiều yêu cầu? Dữ liệu ngoài (thời tiết, giấy phép thi công) có giúp gì không? | 7 |
| **RQ3** (Độ bền khi dữ liệu thay đổi) | Mỗi họ mô hình chịu tác động của điểm gãy 2024 ra sao, và tái huấn luyện định kỳ có giúp thích ứng không? | 6, 7 |
| **RQ4** (Độ tin cậy vận hành) | Khoảng dự báo và cảnh báo tuần cao điểm đáng tin đến mức nào để dùng trong vận hành? | 8 |

## 4. Dữ liệu

| Nguồn | Vai trò | Ghi chú |
|---|---|---|
| SF 311 Cases (`Tree Maintenance`) | Biến mục tiêu: số yêu cầu mỗi khu phố (`nbhd`) mỗi tuần, tính từ thời điểm yêu cầu (`requested_datetime`) | Bộ phân loại yêu cầu đổi giữa ngày 11 và 13/6/2024 |
| SF Street Tree List | Đặc điểm cây theo khu phố: số cây, đường kính thân (`dbh`), tuổi cây tính từ ngày trồng | Chỉ có danh mục hiện tại, không có lịch sử |
| SF Street-Use Permits | Số giấy phép thi công đang hoạt động (`permits_active`) | Biến ngoại sinh, có thể làm tăng va chạm vào cây |
| NOAA GHCN-Daily, trạm SFO | Lượng mưa (`PRCP`), gió (`AWND`), nhiệt độ cao nhất (`TMAX`) | Một trạm, dùng chung cho cả 41 khu phố |

Giai đoạn phân tích: 05/01/2015 – 11/10/2026 (614 tuần × 41 khu phố). Dữ liệu được tải ngày 16/10/2026 và lưu nguyên bản trong `Data/raw/`. Mọi kết quả trong dự án đều tính trên bản dữ liệu này; tải lại từ API vào thời điểm khác có thể cho số liệu hơi khác do thành phố cập nhật dữ liệu liên tục.

## 5. Nguyên tắc chống rò rỉ dữ liệu

Dự án nhấn mạnh tính công bằng của đánh giá, vì vậy mọi bước đều phải tuân theo nguyên tắc: **tại thời điểm dự báo, mô hình chỉ được dùng thông tin đã biết tới thời điểm đó.** Cụ thể:

1. **Đặc điểm cây:** Danh mục cây chỉ có bản hiện tại (tải năm 2026). Với mỗi tuần, chỉ đếm những cây có ngày trồng trước tuần đó, và tính tuổi cây theo chính tuần đó. Cây không có ngày trồng được coi là đã có mặt từ đầu giai đoạn. Cây đã bị chặt bỏ trước năm 2026 không còn trong danh mục nên không thể khôi phục; vì vậy có thêm thử nghiệm bỏ hẳn nhóm đặc trưng này để kiểm tra ảnh hưởng.
2. **Điền giá trị thời tiết thiếu:** chỉ dùng trung vị theo tháng tính trên dữ liệu trước mốc kiểm tra đầu tiên (03/01/2022), nên không dùng thông tin của bất kỳ đoạn kiểm tra nào.
3. **Yêu cầu trùng lặp:** Nhãn "Case is a Duplicate" chỉ có sau khi thành phố xử lý xong yêu cầu. Dự án giữ cả hai phiên bản biến mục tiêu (có và không loại yêu cầu trùng) và báo cáo kết quả trên cả hai.
4. **Chia dữ liệu theo thời gian,** có chừa một tuần đệm giữa tập huấn luyện và tập kiểm tra.
5. **Tinh chỉnh tham số** chỉ dùng tập kiểm định nằm trước mốc kiểm tra, không bao giờ chạm vào tập kiểm tra.

## 6. Các mô hình so sánh

| Họ mô hình | Mô hình | Cần huấn luyện? | Dùng biến ngoài? |
|---|---|---|---|
| Mốc đơn giản | Lặp lại tuần trước; Trung bình 4 tuần; Cùng tuần năm trước | Không | Không |
| Mô hình đếm | Hồi quy Poisson | Có | Có |
| Học máy | Ridge; Random Forest; LightGBM (mục tiêu Poisson); tất cả đều được tinh chỉnh tham số | Có | Có |
| Mô hình nền tảng zero-shot | Chronos-Bolt Small; Chronos-Bolt Base; TimesFM 2.5 | Không | Không |
| Mô hình nền tảng zero-shot có biến ngoài | Chronos-2 (chạy hai cấu hình: chỉ lịch sử nhu cầu, và thêm thời tiết cùng giấy phép thi công) | Không | Có / Không |

## 7. Cách đánh giá

- **Nhiều mốc thời gian:** 9 mốc kiểm tra cách nhau 6 tháng: 03/01/2022, 04/07/2022, 02/01/2023, 03/07/2023, 01/01/2024, 01/07/2024, 06/01/2025, 07/07/2025 và 05/01/2026. Mỗi mốc kiểm tra 26 tuần liên tiếp, các đoạn kiểm tra không chồng lên nhau. Bốn mốc đầu nằm hoàn toàn trước điểm gãy tháng 6/2024, mốc 01/01/2024 chứa điểm gãy, bốn mốc cuối nằm hoàn toàn sau điểm gãy.
- **Cách dự báo:** ở mỗi mốc, các mô hình cần huấn luyện được huấn luyện một lần trên dữ liệu trước mốc (chừa một tuần đệm); sau đó mỗi tuần dự báo số yêu cầu của tuần kế tiếp, dùng dữ liệu đã biết đến tuần hiện tại. Mô hình nền tảng nhận chuỗi lịch sử đến tuần hiện tại làm ngữ cảnh.
- **Thước đo sai số:**
  - MAE và RMSE: sai số tính theo số yêu cầu.
  - MASE: MAE của mô hình chia cho MAE của mốc "lặp lại tuần trước" tính trên tập huấn luyện của cùng khu phố, để so sánh công bằng giữa khu phố ít và nhiều yêu cầu.
  - Báo cáo riêng cho ba nhóm khu phố (thấp, trung bình, cao), chia theo nhu cầu trung bình mỗi tuần trên tập huấn luyện, mỗi nhóm khoảng một phần ba số khu phố.
- **Kiểm định thống kê:** kiểm định Wilcoxon trên sai số trung bình mỗi tuần giữa các cặp mô hình, hiệu chỉnh nhiều phép so sánh bằng phương pháp Holm.
- **Độ tin cậy:** đo độ bao phủ thực tế của khoảng dự báo 80% và 90%, độ rộng khoảng, và điểm số khoảng (interval score). Tuần cao điểm của một khu phố là tuần có số yêu cầu vượt mức phân vị 90% của chính khu phố đó trong 52 tuần trước; cảnh báo được đánh giá bằng precision, recall và F1.

## 8. Quy trình và vai trò từng notebook

| Notebook | Nội dung | Trả lời câu hỏi |
|---|---|---|
| 1. Business Understanding and Analytic Approach | Bài toán, câu hỏi nghiên cứu, phương pháp | — |
| 2. Data Collection Understanding Preparation | Thu thập và làm sạch bốn nguồn dữ liệu; gom theo khu phố và tuần; tính đặc điểm cây theo từng tuần; điền thời tiết thiếu bằng dữ liệu trước tập kiểm tra; chuẩn hoá loại yêu cầu giữa hai bộ phân loại | — |
| 3. Data Analysis with SQL | Thống kê nhu cầu theo khu phố, ngày trong tuần, tháng, năm; tạo đặc trưng trễ và trung bình trượt | RQ1 |
| 4. Data Analysis with Python | Phân tích khám phá, trực quan hoá, tương quan với thời tiết và giấy phép thi công, kiểm định thống kê | RQ1 |
| 5. Model Training and Comparison | Thử nghiệm ban đầu: sáu mô hình trên một mốc kiểm tra (2024–2026) | RQ2 |
| 6. Intake Regime Change | Ghép bộ phân loại cũ và mới; đo mức thay đổi của dữ liệu quanh điểm gãy tháng 6/2024 | RQ1, RQ3 |
| 7. Model Benchmark | So sánh toàn bộ mô hình trên 9 mốc thời gian; thử bỏ từng nhóm đặc trưng; so sánh trước và sau điểm gãy; tái huấn luyện định kỳ | RQ2, RQ3 |
| 8. Uncertainty and Alerts | Khoảng dự báo và cảnh báo tuần cao điểm | RQ4 |

## 9. Kỳ vọng kết quả

- **Tri thức định lượng:** xác định được mô hình nền tảng zero-shot có thật sự vượt mô hình huấn luyện riêng trên dữ liệu đếm thưa của dịch vụ đô thị hay không, và độ bền của từng họ mô hình khi cách ghi nhận dữ liệu thay đổi.
- **Giải pháp hỗ trợ quyết định:** đưa ra dự báo theo tuần cho từng khu phố, kèm khoảng dự báo và cảnh báo sớm, giúp đội bảo trì cây xanh chủ động lên lịch thay vì phản ứng bị động.
- **Sản phẩm học thuật:** bộ mã nguồn Jupyter Notebook chạy lại được trên bản dữ liệu đã lưu, nhật ký sử dụng AI (Audit Log).

## 10. Hạn chế

- Danh mục cây không có lịch sử: cây đã bị chặt bỏ trước năm 2026 không còn trong danh mục, nên đặc điểm cây theo từng tuần chỉ là ước lượng.
- Thời tiết chỉ lấy từ một trạm (sân bay SFO) cho cả thành phố, chưa phản ánh khác biệt giữa các khu phố.
- Số yêu cầu 311 phản ánh mức độ người dân báo cáo, không hoàn toàn là tình trạng thật của cây.
- Nhãn yêu cầu trùng lặp chỉ có sau khi yêu cầu được xử lý, nên biến mục tiêu đã loại yêu cầu trùng không hoàn toàn biết được tại thời điểm dự báo.
- Nghiên cứu chỉ dựa trên dữ liệu của một thành phố; kết quả cần được kiểm chứng thêm ở các thành phố khác.